# Chapter 1 · Qubits and Quantum States

## Objectives

By the end of this notebook the reader will be able to:

1. Mathematically represent a qubit as a vector in $\mathbb{C}^2$.
2. Apply single-qubit gates (H, X, Y, Z, S, T) via matrix multiplication.
3. Calculate measurement probabilities and the Bloch vector.
4. Visualize any single-qubit state on the Bloch sphere.

---

## 1.1 The qubit as a complex vector

The general state of a qubit is written as:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \quad \alpha,\beta \in \mathbb{C}, \quad |\alpha|^2 + |\beta|^2 = 1$$

where the computational basis states are:

$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}, \quad |1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$$

The state $|\psi\rangle$ lives in the Hilbert space $\mathcal{H} = \mathbb{C}^2$ and is determined (up to global phase) by the spherical angles $(\theta, \varphi)$:

$$|\psi\rangle = \cos\frac{\theta}{2}|0\rangle + e^{i\varphi}\sin\frac{\theta}{2}|1\rangle$$

This parametrization defines the **Bloch sphere**.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))  # access to src/

import numpy as np
import matplotlib.pyplot as plt
from src.quantum_math import QuantumMath
from src.quantum_gates import Gates
from src.visualization import QuantumVisualization

print('Modules imported successfully.')

## 1.2 Construction of basic states

In [ ]:
# Base state |0⟩
ket0 = QuantumMath.ket0()
# Base state |1⟩
ket1 = QuantumMath.ket1()
# Superposition |+⟩ = (|0⟩ + |1⟩)/√2
ket_plus = QuantumMath.ket_plus()
# Superposition |-⟩ = (|0⟩ - |1⟩)/√2
ket_minus = QuantumMath.ket_minus()

print(f'|0⟩ = {ket0}')
print(f'|1⟩ = {ket1}')
print(f'|+⟩ = {ket_plus}')
print(f'|-⟩ = {ket_minus}')

# Normalization check
for name, state in [('|0⟩',ket0),('|1⟩',ket1),('|+⟩',ket_plus),('|-⟩',ket_minus)]:
    norm = np.linalg.norm(state)
    print(f'  ||{name}|| = {norm:.6f}')

## 1.3 Single-qubit Gates

A single-qubit quantum gate is a **unitary** matrix $U \in \mathcal{U}(2)$. The most important ones are:

| Gate | Matrix | Effect |
|------|--------|--------|
| $X$ | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | Flip: $|0\rangle\leftrightarrow|1\rangle$ |
| $H$ | $\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ | Uniform superposition |
| $Z$ | $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ | Phase flip |
| $S$ | $\begin{pmatrix}1&0\\0&i\end{pmatrix}$ | Phase $\pi/2$ |
| $T$ | $\begin{pmatrix}1&0\\0&e^{i\pi/4}\end{pmatrix}$ | Phase $\pi/4$ |

In [ ]:
# Verify unitarity of each gate
for name, gate in [('H', Gates.H), ('X', Gates.X), ('Y', Gates.Y),
                   ('Z', Gates.Z), ('S', Gates.S), ('T', Gates.T)]:
    unitary = Gates.is_unitary(gate)
    print(f'Gate {name}: unitary = {unitary}')

print()
# Action of H on |0⟩ → |+⟩
estado_inicial = QuantumMath.ket0()
tras_H = Gates.H @ estado_inicial
print(f'H|0⟩ = {np.round(tras_H, 4)}')

# Double Hadamard recovers the original state: H² = I
de_vuelta = Gates.H @ tras_H
print(f'H²|0⟩ = {np.round(de_vuelta, 4)}')

## 1.4 Measurement probabilities

In [ ]:
# Arbitrary state (defined by Bloch angles)
theta_rad = np.radians(60)   # polar angle
phi_rad   = np.radians(45)   # azimuthal angle

alpha = np.cos(theta_rad / 2)
beta  = np.exp(1j * phi_rad) * np.sin(theta_rad / 2)
psi   = np.array([alpha, beta])

print(f'State |ψ⟩ with θ=60°, φ=45°:')
print(f'  α = {alpha:.4f}')
print(f'  β = {beta.real:.4f} + {beta.imag:.4f}i')

probs = QuantumMath.probabilities(psi)
print(f'  P(|0⟩) = |α|² = {probs[0]:.4f}')
print(f'  P(|1⟩) = |β|² = {probs[1]:.4f}')
print(f'  Sum    = {np.sum(probs):.6f}')

# Simulated measurement
counts = QuantumMath.measure(psi, n_shots=2048)
print(f'\nResults of 2048 measurements: {counts}')

In [ ]:
# Histogram visualization of measurements
fig = QuantumVisualization.plot_histogram(
    counts,
    title=f'Measurement distribution (θ=60°, φ=45°)',
)
plt.show()

## 1.5 Bloch vector and 3D visualization

In [ ]:
# Calculate the Bloch vector
bv = QuantumMath.bloch_vector(psi)
print(f'Bloch vector for |ψ⟩:')
print(f'  x = {bv[0]:.4f}')
print(f'  y = {bv[1]:.4f}')
print(f'  z = {bv[2]:.4f}')
print(f'  ||r|| = {np.sqrt(sum(c**2 for c in bv)):.6f}  (should be 1 for a pure state)')

# Static visualization in matplotlib
fig = QuantumVisualization.plot_bloch_vector(psi, title='State |ψ⟩ (θ=60°, φ=45°)')
plt.show()

## 1.6 Equivalent circuit in Qiskit

We now replicate the same experiment with Qiskit to verify consistency.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

# Prepare state |ψ⟩ with Qiskit (using initialize)
qc = QuantumCircuit(1)
qc.initialize([alpha, beta], 0)
qc.measure_all()

# Simulation
sim_backend = AerSimulator()
qc_transpiled = qc
job = sim_backend.run(qc, shots=2048)
result = job.result()
counts_qiskit = result.get_counts()

print('Qiskit counts:', counts_qiskit)

# Circuit diagram
qc_draw = QuantumCircuit(1)
qc_draw.initialize([alpha, beta], 0)
print(qc_draw.draw('text'))

## 1.7 Proposed exercises

1. Construct the state $|y+\rangle = \frac{|0\rangle + i|1\rangle}{\sqrt{2}}$ and calculate its Bloch vector. On which axis of the sphere does it lie?

2. Prove algebraically that $H X H = Z$. Verify it computationally.

3. Apply the sequence $S \rightarrow T \rightarrow H$ to $|0\rangle$ and represent the resulting state on the Bloch sphere.

4. How many times must gate $T$ be applied so that $T^n = I$? Check it numerically.

5. Calculate the fidelity between $|+\rangle$ and the state obtained by applying $Rx(\pi/6)$ to $|0\rangle$.

In [ ]:
# Workspace for exercises
# ─────────────────────────────────────

# Exercise 2: HXH = Z
HXH = Gates.H @ Gates.X @ Gates.H
print('HXH =\n', np.round(HXH, 4))
print('Z   =\n', Gates.Z)
print('Are they equal?', np.allclose(HXH, Gates.Z))